## Setup

In [1]:
import json
import sqlite3
import re
import time
import warnings
import logging
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM

from langchain_groq import ChatGroq
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import requests
import os

from dotenv import load_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

load_dotenv()

d:\GenAI\Task\Nosh_Robotics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:03<00:00, 87.97it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [3]:
def normalize_batch_gpu(ingredients: List[str], batch_size: int = 128) -> Dict[str, str]:
    results = {}
    
    for i in range(0, len(ingredients), batch_size):
        batch = ingredients[i:i+batch_size]
        
        prompts = [
            f"""Extract base ingredient. Remove quantities and preparation words.

Examples:
"2 cups all-purpose flour" → all-purpose flour
"1/4 cup minced garlic" → garlic
"4 skinless, boneless chicken breast halves" → chicken breast

Input: "{ing}"
Output:"""
            for ing in batch
        ]
        
        try:
            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128
            )
            
            if device == "cuda":
                inputs = {k: v.cuda() for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=8,
                    temperature=0.1,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    num_beams=1
                )
            
            for idx, (ing, output) in enumerate(zip(batch, outputs)):
                input_len = inputs["input_ids"][idx].shape[0]
                result = tokenizer.decode(output[input_len:], skip_special_tokens=True)
                
                normalized = result.strip().lower()
                
                if normalized.startswith("["):
                    try:
                        parsed = json.loads(normalized)
                        if isinstance(parsed, list) and len(parsed) > 0:
                            normalized = str(parsed[0]).lower()
                    except:
                        normalized = normalized.strip('[]"\'')
                
                normalized = normalized.split("\n")[0].strip()
                results[ing] = normalized if normalized else ing.lower().strip()
            
            if device == "cuda" and i % 1000 == 0:
                torch.cuda.empty_cache()
        
        except Exception as e:
            for ing in batch:
                results[ing] = ing.lower().strip()
    
    return results

def build_normalization_cache(all_ingredients: List[str], batch_size: int = 128) -> Dict[str, str]:
    unique = list(set(all_ingredients))
    cache = {}
    
    chunk_size = 1000
    chunks = [unique[i:i+chunk_size] for i in range(0, len(unique), chunk_size)]
    
    print(f"Normalizing {len(unique):,} unique ingredients...")
    start_time = time.time()
    
    for idx, chunk in enumerate(chunks):
        chunk_result = normalize_batch_gpu(chunk, batch_size=batch_size)
        cache.update(chunk_result)
        
        if (idx + 1) % 5 == 0 or idx == len(chunks) - 1:
            done = min((idx + 1) * chunk_size, len(unique))
            elapsed = time.time() - start_time
            rate = done / elapsed if elapsed > 0 else 0
            print(f"Progress: {done:,}/{len(unique):,} ({100*done/len(unique):.1f}%) | Rate: {rate:.0f} ing/s")
    
    total_time = time.time() - start_time
    print(f"Normalization complete: {len(cache):,} entries in {total_time/60:.1f} minutes")
    
    return cache

def normalize_ingredient(ing: str, cache: Dict[str, str] = None) -> str:
    if cache and ing in cache:
        return cache[ing]
    return ing.lower().strip()

def clean_ingredient(ing: str) -> str:
    return ing.replace("ADVERTISEMENT", "").strip()

def count_steps(instructions: str) -> int:
    if not instructions:
        return 0
    steps = [s.strip() for s in re.split(r"\n|(?<!\d)\.(?!\d)", instructions) if s.strip()]
    return len(steps)

## Database Setup

In [4]:
def load_recipes_to_sqlite(db_path: str = "recipes.db") -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS recipes (
            id TEXT PRIMARY KEY,
            source TEXT,
            title TEXT,
            ingredients_json TEXT,
            instructions TEXT,
            step_count INTEGER,
            ingredient_count INTEGER
        )
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS ingredients (
            recipe_id TEXT,
            ingredient_normalized TEXT,
            ingredient_original TEXT,
            FOREIGN KEY(recipe_id) REFERENCES recipes(id)
        )
    """)
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_ing_norm ON ingredients(ingredient_normalized)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_recipe_id ON ingredients(recipe_id)")

    json_files = [
        ("recipes_raw/recipes_raw_nosource_ar.json", "allrecipes"),
        ("recipes_raw/recipes_raw_nosource_epi.json", "epicurious"),
        ("recipes_raw/recipes_raw_nosource_fn.json", "foodnetwork"),
    ]

    all_raw: List[tuple] = []
    all_raw_ings: List[str] = []

    for json_file, source in json_files:
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"  {source}: {len(data):,} recipes")
        for recipe_id, recipe in data.items():
            ings = [
                clean_ingredient(ing)
                for ing in recipe.get("ingredients", [])
                if ing.strip() and ing.strip() != "ADVERTISEMENT"
            ]
            all_raw.append((source, recipe_id, recipe, ings))
            all_raw_ings.extend(ings)

    norm_cache = build_normalization_cache(all_raw_ings)

    total = 0
    for source, recipe_id, recipe, ingredients in all_raw:
        title = recipe.get("title", f"Recipe {recipe_id[:8]}")

        cursor.execute("""
            INSERT OR REPLACE INTO recipes
                (id, source, title, ingredients_json, instructions, step_count, ingredient_count)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            recipe_id, source, title,
            json.dumps(ingredients),
            recipe.get("instructions", ""),
            count_steps(recipe.get("instructions", "")),
            len(ingredients),
        ))

        for ingredient in ingredients:
            normalized = normalize_ingredient(ingredient, norm_cache)
            if normalized:
                cursor.execute("""
                    INSERT INTO ingredients (recipe_id, ingredient_normalized, ingredient_original)
                    VALUES (?, ?, ?)
                """, (recipe_id, normalized, ingredient))

        total += 1
        if total % 10000 == 0:
            conn.commit()

    conn.commit()
    
    cursor.execute("SELECT COUNT(*) FROM recipes")
    recipe_count = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(DISTINCT ingredient_normalized) FROM ingredients")
    ing_count = cursor.fetchone()[0]
    
    print(f"\nDatabase ready: {recipe_count:,} recipes, {ing_count:,} unique ingredients")
    return conn

DB_PATH = "recipes.db"

if Path(DB_PATH).exists():
    print("Using existing database...")
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM recipes")
    print(f"✅ Database loaded: {cursor.fetchone()[0]:,} recipes")
else:
    conn = load_recipes_to_sqlite(DB_PATH)

Using existing database...
✅ Database loaded: 125,164 recipes


## Hybrid Search Engine

In [5]:
@dataclass
class RecipeDocument:
    recipe_id: str
    page_content: str
    metadata: dict
    score: float = 0.0

class HybridRecipeSearchEngine:
    def __init__(
        self,
        conn: sqlite3.Connection,
        llm,
        dense_weight: float = 0.6,
        sparse_weight: float = 0.4,
    ):
        self.conn = conn
        self.llm = llm
        self.dense_weight = dense_weight
        self.sparse_weight = sparse_weight

        self.embeddings = SentenceTransformer('BAAI/bge-small-en-v1.5')
        self.reranker = CrossEncoder('BAAI/bge-reranker-base', max_length=512)
        self.documents = self._load_all_recipes()
        self._build_bm25_index()
        self._build_dense_index()
    
    def _load_all_recipes(self) -> List[RecipeDocument]:
        cursor = self.conn.execute("""
            SELECT id, title, ingredients_json, instructions
            FROM recipes
            WHERE ingredient_count > 0 OR step_count > 0
        """)
        
        documents = []
        for recipe_id, title, ingredients_json, instructions in cursor.fetchall():
            ingredients = json.loads(ingredients_json)
            ingredients_text = ", ".join(ingredients)
            
            page_content = f"{title}. Ingredients: {ingredients_text}"
            if instructions:
                page_content += f". {instructions[:200]}"
            
            documents.append(RecipeDocument(
                recipe_id=recipe_id,
                page_content=page_content,
                metadata={
                    "title": title,
                    "ingredient_count": len(ingredients)
                }
            ))
        
        return documents
    
    def _build_bm25_index(self):
        tokenized_corpus = [
            doc.page_content.lower().split()
            for doc in self.documents
        ]
        self.bm25 = BM25Okapi(tokenized_corpus)
    
    def _build_dense_index(self):
        EMBEDDINGS_FILE = "embeddings.npy"
        
        if Path(EMBEDDINGS_FILE).exists():
            self.dense_embeddings = np.load(EMBEDDINGS_FILE)
        else:
            texts = [doc.page_content for doc in self.documents]
            self.dense_embeddings = self.embeddings.encode(
                texts,
                show_progress_bar=True,
                batch_size=512,
                normalize_embeddings=True
            )
            np.save(EMBEDDINGS_FILE, self.dense_embeddings)
    
    def _hyde_generate(self, query: str) -> str:
        hyde_prompt = f"""Given this query: "{query}"
        
Write a detailed recipe description (150-200 words) that would match this request.
Include typical ingredients and cooking methods. Write as if this recipe already exists.

Do NOT mention the query itself. Just write the recipe description."""
        
        try:
            response = self.llm.invoke(hyde_prompt)
            return response.content
        except:
            return query
    
    def _dense_search(self, query: str, k: int = 15, use_hyde: bool = True) -> List[RecipeDocument]:
        if use_hyde:
            search_text = self._hyde_generate(query)
        else:
            search_text = query
        
        query_embedding = self.embeddings.encode([search_text], normalize_embeddings=True)[0]
        scores = np.dot(self.dense_embeddings, query_embedding)
        top_indices = np.argsort(scores)[::-1][:k]
        
        results = []
        for idx in top_indices:
            doc = self.documents[idx]
            doc.score = scores[idx]
            results.append(doc)
        
        return results
    
    def _sparse_search(self, query: str, k: int = 10) -> List[RecipeDocument]:
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:k]
        
        results = []
        for idx in top_indices:
            doc = self.documents[idx]
            doc.score = scores[idx]
            results.append(doc)
        
        return results
    
    def _ensemble_fusion(self, dense_docs: List[RecipeDocument], sparse_docs: List[RecipeDocument]) -> List[RecipeDocument]:
        doc_scores = {}
        
        for rank, doc in enumerate(dense_docs):
            doc_scores[doc.recipe_id] = doc_scores.get(doc.recipe_id, 0) + \
                self.dense_weight * (1.0 / (rank + 1))
        
        for rank, doc in enumerate(sparse_docs):
            doc_scores[doc.recipe_id] = doc_scores.get(doc.recipe_id, 0) + \
                self.sparse_weight * (1.0 / (rank + 1))
        
        sorted_ids = sorted(doc_scores.keys(), key=lambda x: doc_scores[x], reverse=True)
        
        doc_map = {doc.recipe_id: doc for doc in dense_docs + sparse_docs}
        results = []
        for rid in sorted_ids:
            if rid in doc_map:
                doc = doc_map[rid]
                doc.score = doc_scores[rid]
                results.append(doc)
        
        return results
    
    def _rerank(self, query: str, documents: List[RecipeDocument], top_n: int = 5) -> List[RecipeDocument]:
        if not documents:
            return []
        
        pairs = [[query, doc.page_content] for doc in documents]
        scores = self.reranker.predict(pairs, show_progress_bar=False)
        
        scored_docs = list(zip(documents, scores))
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        
        results = []
        for doc, score in scored_docs[:top_n]:
            doc.score = score
            results.append(doc)
        
        return results
    
    def search(
        self,
        query: str,
        k: int = 5,
        use_hyde: bool = True,
        use_reranking: bool = True,
        valid_recipe_ids: Optional[set] = None
    ) -> List[Dict]:
        if valid_recipe_ids is not None:
            valid_indices = [i for i, doc in enumerate(self.documents) if doc.recipe_id in valid_recipe_ids]
            if not valid_indices:
                return []
            
            if use_hyde:
                search_text = self._hyde_generate(query)
            else:
                search_text = query
            query_embedding = self.embeddings.encode([search_text], normalize_embeddings=True)[0]
            valid_embeddings = self.dense_embeddings[valid_indices]
            scores = np.dot(valid_embeddings, query_embedding)
            top_k_in_valid = np.argsort(scores)[::-1][:15]
            dense_docs = [self.documents[valid_indices[idx]] for idx in top_k_in_valid]
            for idx, doc in zip(top_k_in_valid, dense_docs):
                doc.score = scores[idx]
            
            tokenized_query = query.lower().split()
            all_scores = self.bm25.get_scores(tokenized_query)
            valid_scores = sorted([(i, all_scores[i]) for i in valid_indices], key=lambda x: x[1], reverse=True)[:10]
            sparse_docs = [self.documents[idx] for idx, _ in valid_scores]
            for (idx, score), doc in zip(valid_scores, sparse_docs):
                doc.score = score
        else:
            dense_docs = self._dense_search(query, k=15, use_hyde=use_hyde)
            sparse_docs = self._sparse_search(query, k=10)
        
        fused_docs = self._ensemble_fusion(dense_docs, sparse_docs)
        if use_reranking:
            final_docs = self._rerank(query, fused_docs[:20], top_n=k)
        else:
            final_docs = fused_docs[:k]
        return self._fetch_recipe_details([doc.recipe_id for doc in final_docs])
    
    def _fetch_recipe_details(self, recipe_ids: List[str]) -> List[Dict]:
        if not recipe_ids:
            return []
        
        placeholders = ','.join('?' * len(recipe_ids))
        cursor = self.conn.execute(f"""
            SELECT id, title, ingredients_json, instructions, step_count
            FROM recipes
            WHERE id IN ({placeholders})
        """, recipe_ids)
        
        results_map = {}
        for row in cursor.fetchall():
            results_map[row[0]] = {
                'id': row[0],
                'title': row[1],
                'ingredients': json.loads(row[2]),
                'instructions': row[3],
                'step_count': row[4]
            }
        
        return [results_map[rid] for rid in recipe_ids if rid in results_map]


## LLM Setup & Search Engine Initialization

In [6]:
class OmnirouteClaude(BaseChatModel):
    """Custom LangChain wrapper for local Omniroute API"""
    
    base_url: str = "http://localhost:20128/v1/chat/completions"
    model: str = "no-think/antigravity/claude-sonnet-4-6"
    temperature: float = 0.0
    
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        api_messages = []
        for msg in messages:
            if hasattr(msg, 'type'):
                role = 'user' if msg.type == 'human' else 'assistant'
                if msg.type == 'system':
                    role = 'system'
            else:
                role = 'user'
            
            api_messages.append({
                "role": role,
                "content": msg.content if hasattr(msg, 'content') else str(msg)
            })
        
        payload = {
            "model": self.model,
            "messages": api_messages,
            "temperature": self.temperature,
            "max_tokens": kwargs.get("max_tokens", 2048)
        }
        
        try:
            response = requests.post(
                self.base_url,
                headers={"Content-Type": "application/json"},
                json=payload,
                timeout=30
            )
            response.raise_for_status()
            result = response.json()
            
            content = result["choices"][0]["message"]["content"]
            
            from langchain_core.outputs import ChatGeneration, ChatResult
            
            generation = ChatGeneration(message=AIMessage(content=content))
            return ChatResult(generations=[generation])
            
        except requests.exceptions.RequestException as e:
            print(f"Omniroute API error: {e}")
            raise
    
    @property
    def _llm_type(self):
        return "omniroute-claude"
    
    @property
    def _identifying_params(self):
        return {"model": self.model}

In [7]:
groq_llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

try:
    claude_llm = OmnirouteClaude(
        model="no-think/antigravity/claude-sonnet-4-6",
        temperature=0
    )
except Exception as e:
    claude_llm = ChatAnthropic(model="claude-3-5-sonnet-20241022", temperature=0)

In [8]:
search_engine = HybridRecipeSearchEngine(
    conn=conn,
    llm=groq_llm,
    dense_weight=0.6,
    sparse_weight=0.4
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 928.57it/s]


## Conversation State

In [9]:
@dataclass
class ConversationState:
    exclude_ingredients: List[str] = field(default_factory=list)
    include_ingredients: List[str] = field(default_factory=list)
    max_steps: Optional[int] = None
    user_ingredients: List[str] = field(default_factory=list)
    last_results: List[str] = field(default_factory=list)
    last_query: str = ""
    
    def update_from_query(self, query: str):
        query_lower = query.lower()
        
        if any(phrase in query_lower for phrase in ['no ', 'without ', 'cannot have ', 'exclude ', 'allergic to ']):
            for word in ['dairy', 'milk', 'cream', 'butter', 'cheese', 'peanut', 'nut', 'egg', 'gluten', 'wheat']:
                if word in query_lower and word not in self.exclude_ingredients:
                    self.exclude_ingredients.append(word)
        
        step_match = re.search(r'(at most|max|maximum|under|less than)\s+(\d+)\s+(?:\w+\s+)*steps?', query_lower)
        if step_match:
            self.max_steps = int(step_match.group(2))
    
    def to_dict(self) -> dict:
        return {
            "exclude_ingredients": self.exclude_ingredients,
            "include_ingredients": self.include_ingredients,
            "max_steps": self.max_steps,
            "user_ingredients": self.user_ingredients
        }

## Task 1: Recipe Search & Analytics

In [10]:
def filter_by_state(recipe_ids: List[str], state: ConversationState) -> List[str]:
    if not state.exclude_ingredients and state.max_steps is None:
        return recipe_ids
    
    filtered = set(recipe_ids)
    
    if state.exclude_ingredients:
        placeholders = ','.join('?' * len(state.exclude_ingredients))
        cursor = conn.execute(f"""
            SELECT DISTINCT recipe_id FROM ingredients
            WHERE ingredient_normalized IN ({placeholders})
        """, state.exclude_ingredients)
        excluded = {row[0] for row in cursor.fetchall()}
        filtered -= excluded
    
    if state.max_steps:
        cursor = conn.execute("""
            SELECT id FROM recipes WHERE step_count > ?
        """, (state.max_steps,))
        excluded = {row[0] for row in cursor.fetchall()}
        filtered -= excluded
    
    return [rid for rid in recipe_ids if rid in filtered]

def search_recipes(query: str, state: ConversationState, k: int = 5) -> dict:
    valid_ids = None
    if state.exclude_ingredients or state.max_steps:
        cursor = conn.execute("SELECT id FROM recipes")
        valid_ids = {row[0] for row in cursor.fetchall()}
        
        if state.exclude_ingredients:
            placeholders = ','.join('?' * len(state.exclude_ingredients))
            cursor = conn.execute(f"SELECT DISTINCT recipe_id FROM ingredients WHERE ingredient_normalized IN ({placeholders})", state.exclude_ingredients)
            excluded = {row[0] for row in cursor.fetchall()}
            valid_ids -= excluded
        
        if state.max_steps:
            cursor = conn.execute("SELECT id FROM recipes WHERE step_count > ?", (state.max_steps,))
            excluded = {row[0] for row in cursor.fetchall()}
            valid_ids -= excluded
    
    results = search_engine.search(query, k=k, valid_recipe_ids=valid_ids)
    return {
        'recipes': results,
        'total_found': len(results),
        'filters_applied': state.to_dict()
    }

def parse_stats_query(query: str) -> dict:
    prompt = f"""Extract from this recipe analytics query:
- ingredients: list of ingredient names mentioned (empty list if none)
- operation: one of 'count', 'rank_steps', 'common', 'average'
- filters: any additional constraints

Query: "{query}"

Return ONLY valid JSON: {{"ingredients": [], "operation": "", "filters": {{}}}}"""
    
    try:
        response = groq_llm.invoke(prompt)
        content = response.content.strip()
        if content.startswith("```"):
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
        content = content.strip()
        parsed = json.loads(content)
        return parsed
    except:
        return {"ingredients": [], "operation": "unknown", "filters": {}}

def compute_statistics(query: str) -> str:
    parsed = parse_stats_query(query)
    
    if parsed['operation'] == 'count' and len(parsed['ingredients']) >= 1:
        ingredients = [normalize_ingredient(ing) for ing in parsed['ingredients']]
        
        if len(ingredients) >= 2:
            placeholders = ' INTERSECT '.join([
                "SELECT recipe_id FROM ingredients WHERE ingredient_normalized = ?"
                for _ in ingredients
            ])
            cursor = conn.execute(f"SELECT COUNT(*) FROM ({placeholders})", ingredients)
            count = cursor.fetchone()[0]
            cursor = conn.execute("SELECT COUNT(*) FROM recipes")
            total = cursor.fetchone()[0]
            pct = round(count / total * 100, 2) if total > 0 else 0
            return (
                f"Found {count:,} recipes containing all of: {', '.join(ingredients)}.\n\n"
                f"This represents {pct}% of the {total:,} total recipes in the dataset."
            )
        else:
            cursor = conn.execute(
                "SELECT COUNT(DISTINCT recipe_id) FROM ingredients WHERE ingredient_normalized = ?",
                (ingredients[0],)
            )
            count = cursor.fetchone()[0]
            cursor = conn.execute("SELECT COUNT(*) FROM recipes")
            total = cursor.fetchone()[0]
            pct = round(count / total * 100, 2) if total > 0 else 0
            return f"Found {count:,} recipes containing '{ingredients[0]}' ({pct}% of {total:,} total recipes)."
    
    if parsed['operation'] == 'rank_steps' and len(parsed['ingredients']) >= 1:
        ingredient = normalize_ingredient(parsed['ingredients'][0])
        cursor = conn.execute("""
            SELECT r.id, r.title, r.step_count
            FROM recipes r
            JOIN ingredients i ON r.id = i.recipe_id
            WHERE i.ingredient_normalized = ?
            ORDER BY r.step_count ASC
            LIMIT 5
        """, (ingredient,))
        results = cursor.fetchall()
        if results:
            output = [f"Top 5 recipes with '{ingredient}' by fewest steps:\n"]
            for i, (recipe_id, title, steps) in enumerate(results, 1):
                output.append(f"{i}. {title} ({steps} steps)")
            return '\n'.join(output)
        else:
            return f"No recipes found with '{ingredient}'."
    
    return "I can compute statistics like recipe counts, common ingredients, and rankings. Please ask a specific analytical question."

## Task 2: Ingredient Matching

In [11]:
def score_recipe_match(recipe_id: str, user_ingredients: List[str], hard_constraints: dict) -> Optional[dict]:
    cursor = conn.execute("""
        SELECT ingredient_normalized, ingredient_original
        FROM ingredients
        WHERE recipe_id = ?
    """, (recipe_id,))
    recipe_ings = [(row[0], row[1]) for row in cursor.fetchall()]
    recipe_norm = {ing[0] for ing in recipe_ings}
    
    user_norm = {normalize_ingredient(ing) for ing in user_ingredients}
    
    excluded = set(hard_constraints.get('exclude', []))
    if recipe_norm & excluded:
        return None
    
    max_steps = hard_constraints.get('max_steps')
    if max_steps:
        cursor = conn.execute("SELECT step_count FROM recipes WHERE id = ?", (recipe_id,))
        step_count = cursor.fetchone()[0]
        if step_count > max_steps:
            return None
    
    matched = user_norm & recipe_norm
    missing = recipe_norm - user_norm
    
    score = len(matched) / len(recipe_ings) 
    
    return {
        'recipe_id': recipe_id,
        'score': score,
        'matched': list(matched),
        'missing': list(missing),
        'match_count': len(matched),
        'missing_count': len(missing)
    }

def match_recipes(user_ingredients: List[str], hard_constraints: dict, k: int = 5) -> List[dict]:
    user_norm = [normalize_ingredient(ing) for ing in user_ingredients]
    placeholders = ','.join('?' * len(user_norm))
    
    cursor = conn.execute(f"""
        SELECT DISTINCT recipe_id
        FROM ingredients
        WHERE ingredient_normalized IN ({placeholders})
    """, user_norm)
    
    candidate_ids = [row[0] for row in cursor.fetchall()]
    
    scored = []
    for recipe_id in candidate_ids:
        match = score_recipe_match(recipe_id, user_ingredients, hard_constraints)
        if match:
            scored.append(match)
    
    scored.sort(key=lambda x: x['score'], reverse=True)
    
    top_k = scored[:k]
    recipe_ids = [match['recipe_id'] for match in top_k]
    
    placeholders = ','.join('?' * len(recipe_ids))
    cursor = conn.execute(f"""
        SELECT id, title, ingredients_json, instructions, step_count
        FROM recipes
        WHERE id IN ({placeholders})
    """, recipe_ids)
    
    recipes = []
    for row in cursor.fetchall():
        recipes.append({
            'id': row[0],
            'title': row[1],
            'ingredients': json.loads(row[2]),
            'instructions': row[3],
            'step_count': row[4]
        })
    
    for recipe, match in zip(recipes, top_k):
        recipe.update(match)
    
    return recipes

## Task 3: Evidence-Based Substitution

In [12]:
def find_substitution_evidence(recipe_id: str, ingredient_to_replace: str, restrictions: List[str]) -> dict:
    cursor = conn.execute("SELECT title FROM recipes WHERE id = ?", (recipe_id,))
    title = cursor.fetchone()[0].lower()
    
    category_keywords = {
        'baking': ['bread', 'cake', 'cookie', 'muffin', 'pie', 'pastry', 'biscuit'],
        'pasta': ['pasta', 'spaghetti', 'penne', 'fettuccine', 'lasagna'],
        'dessert': ['dessert', 'pudding', 'ice cream', 'chocolate', 'sweet'],
    }
    
    category = 'general'
    for cat, keywords in category_keywords.items():
        if any(kw in title for kw in keywords):
            category = cat
            break
    
    ing_norm = normalize_ingredient(ingredient_to_replace)
    
    cursor = conn.execute("""
        SELECT r.id, r.title
        FROM recipes r
        WHERE r.id != ?
        AND r.id NOT IN (
            SELECT recipe_id FROM ingredients WHERE ingredient_normalized = ?
        )
        LIMIT 200
    """, (recipe_id, ing_norm))
    
    similar_recipes = [row[0] for row in cursor.fetchall()]
    
    if similar_recipes:
        placeholders = ','.join('?' * len(similar_recipes))
        cursor = conn.execute(f"""
            SELECT ingredient_normalized, COUNT(*) as freq
            FROM ingredients
            WHERE recipe_id IN ({placeholders})
            GROUP BY ingredient_normalized
            ORDER BY freq DESC
            LIMIT 50
        """, similar_recipes)
        candidates = cursor.fetchall()
        
        cursor = conn.execute("SELECT ingredient_normalized, COUNT(*) as total FROM ingredients GROUP BY ingredient_normalized")
        baseline = {row[0]: row[1] for row in cursor.fetchall()}
        total_recipes = 125000
        without_count = len(similar_recipes)
        
        scored_candidates = []
        for ing, freq in candidates:
            baseline_freq = baseline.get(ing, 0)
            if baseline_freq > 0:
                enrichment = (freq / without_count) / (baseline_freq / total_recipes)
                scored_candidates.append((ing, freq, enrichment))
        
        scored_candidates.sort(key=lambda x: x[2], reverse=True)
    else:
        scored_candidates = []
    
    restrictions_norm = [normalize_ingredient(r) for r in restrictions]
    scored_candidates = [(ing, freq, score) for ing, freq, score in scored_candidates if ing not in restrictions_norm]
    top_substitutes = scored_candidates[:3]
    
    return {
        'original': ingredient_to_replace,
        'category': category,
        'candidates': [(ing, freq) for ing, freq, _ in top_substitutes],
        'candidates_with_scores': top_substitutes,
        'evidence_count': len(similar_recipes)
    }

def suggest_substitution(recipe_id: str, ingredient_to_replace: str, restrictions: List[str]) -> str:
    evidence = find_substitution_evidence(recipe_id, ingredient_to_replace, restrictions)
    
    placeholders = ','.join(['?'])
    cursor = conn.execute(f"""
        SELECT id, title, ingredients_json, instructions, step_count
        FROM recipes
        WHERE id = ?
    """, (recipe_id,))
    
    row = cursor.fetchone()
    if not row:
        return "Recipe not found."
    
    recipe = {
        'id': row[0],
        'title': row[1],
        'ingredients': json.loads(row[2]),
        'instructions': row[3],
        'step_count': row[4]
    }
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a culinary expert specializing in recipe adaptation."),
        ("human", """I need to adapt this recipe by replacing an ingredient.

Recipe: {title}
Category: {category}
Ingredient to replace: {ingredient}
Restrictions: {restrictions}

Evidence from dataset:
Analyzed {evidence_count} similar recipes without '{ingredient}'.
Most common alternatives found:
{candidates}

Original ingredients:
{ingredients}

Original instructions:
{instructions}

Please provide:
1. Best substitute recommendation (choose from evidence OR explain if none suitable)
2. Updated ingredient list
3. Modified instructions (if needed)
4. Expected impact on taste/texture

Ground your recommendation in the dataset evidence above.""")
    ])
    
    candidates_with_scores = evidence.get('candidates_with_scores', evidence['candidates'])
    if candidates_with_scores and len(candidates_with_scores[0]) == 3:
        candidates_text = '\n'.join([f"- {ing}: appears in {freq} similar recipes (enrichment: {score:.2f}x)" for ing, freq, score in candidates_with_scores])
    else:
        candidates_text = '\n'.join([f"- {ing}: appears in {freq} similar recipes" for ing, freq in evidence['candidates']])
    if not candidates_text:
        candidates_text = "No strong alternatives found in dataset (limited evidence)."
    
    chain = prompt | claude_llm | StrOutputParser()
    
    response = chain.invoke({
        "title": recipe['title'],
        "category": evidence['category'],
        "ingredient": ingredient_to_replace,
        "restrictions": ', '.join(restrictions) if restrictions else 'None',
        "evidence_count": evidence['evidence_count'],
        "candidates": candidates_text,
        "ingredients": '\n'.join([f"- {ing}" for ing in recipe['ingredients']]),
        "instructions": recipe['instructions']
    })
    
    return response

## Intent Classification & Main Chat Loop

In [13]:
def classify_intent(query: str) -> str:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """Classify the query into ONE of these types:
SEARCH - looking for recipes
STATS - asking for counts, rankings, analytics
MATCH - has ingredients, wants suggestions
SUBSTITUTE - wants to replace an ingredient

Respond with ONLY the type name."""),
        ("human", "{query}")
    ])
    
    chain = prompt | groq_llm | StrOutputParser()
    response = chain.invoke({"query": query})
    return response.strip().upper()

def process_query(query: str, state: ConversationState) -> str:
    state.update_from_query(query)
    state.last_query = query
    
    intent = classify_intent(query)
    
    try:
        if intent == 'SEARCH':
            result = search_recipes(query, state)
            recipes = result['recipes']
            
            if not recipes:
                return f"""No recipes found matching your criteria.
                
Active filters: {state.to_dict()}

Try relaxing your constraints or use different keywords."""
            
            state.last_results = [r['id'] for r in recipes]
            
            output = [f"Found {result['total_found']} recipes. Showing top {len(recipes)}:\n"]
            for i, recipe in enumerate(recipes, 1):
                output.append(f"{i}. **{recipe['title']}**")
                output.append(f"   Steps: {recipe['step_count']}")
                output.append(f"   Ingredients: {', '.join(recipe['ingredients'][:5])}...\n")
            
            if state.exclude_ingredients or state.max_steps:
                output.append(f"\nActive filters: {state.to_dict()}")
            
            return '\n'.join(output)
        
        elif intent == 'STATS':
            return compute_statistics(query)
        
        elif intent == 'MATCH':
            query_lower = query.lower()
            if 'i have' in query_lower:
                ingredients_text = query_lower.split('i have')[1].split('.')[0]
                ingredients = [ing.strip() for ing in re.split(r',|and', ingredients_text) if ing.strip()]
                state.user_ingredients = ingredients
            
            if not state.user_ingredients:
                return "Please tell me what ingredients you have. For example: 'I have eggs, flour, and milk.'"
            
            hard_constraints = {
                'exclude': state.exclude_ingredients,
                'max_steps': state.max_steps
            }
            
            matches = match_recipes(state.user_ingredients, hard_constraints)
            
            if not matches:
                return f"""No recipes found matching your ingredients and constraints.
                
Your ingredients: {', '.join(state.user_ingredients)}
Constraints: {hard_constraints}

Try adding more ingredients or relaxing constraints."""
            
            output = [f"Top {len(matches)} recipes using your ingredients:\n"]
            for i, recipe in enumerate(matches, 1):
                output.append(f"{i}. **{recipe['title']}** (Match Score: {recipe['score']:.1f})")
                output.append(f"   Matched: {', '.join(recipe['matched'][:5])} ({recipe['match_count']} total)")
                output.append(f"   Missing: {', '.join(recipe['missing'][:3])} ({recipe['missing_count']} total)")
                output.append(f"   Steps: {recipe['step_count']}\n")
            
            return '\n'.join(output)
        
        elif intent == 'SUBSTITUTE':
            if not state.last_results:
                return "Please search for a recipe first, then ask about substitutions."
            
            recipe_id = state.last_results[0]
            
            query_lower = query.lower()
            ingredient = None
            for word in ['egg', 'milk', 'cream', 'butter', 'flour', 'sugar', 'oil', 'cheese']:
                if word in query_lower:
                    ingredient = word
                    break
            
            if not ingredient:
                return "Please specify which ingredient to replace (e.g., 'replace eggs' or 'without milk')."
            
            restrictions = state.exclude_ingredients
            return suggest_substitution(recipe_id, ingredient, restrictions)
        
        else:
            return "I can help you search recipes, compute statistics, match recipes to ingredients, or suggest substitutions. What would you like to do?"
    
    except Exception as e:
        return f"Error processing query: {str(e)}"

## Interactive Chat Interface

In [14]:
def chat():
    state = ConversationState()
    
    print("="*80)
    print("Recipe Intelligence Chatbot - Type 'quit' to exit, 'reset' to clear state")
    print("="*80)
    print("\nCapabilities:")
    print("  • Search recipes with semantic understanding")
    print("  • Compute statistics and rankings")
    print("  • Match recipes to your available ingredients")
    print("  • Suggest evidence-based ingredient substitutions\n")
    
    while True:
        user_input = input("\n🧑 You: ").strip()
        
        if not user_input:
            continue
        
        if user_input.lower() == 'quit':
            print("\nGoodbye!")
            break
        
        if user_input.lower() == 'reset':
            state = ConversationState()
            print("\n✅ Conversation state reset.")
            continue
        
        if user_input.lower() == 'state':
            print(f"\nCurrent state: {state.to_dict()}")
            continue
        
        response = process_query(user_input, state)
        print(f"\n🤖 Bot:\n{response}")

chat()

Recipe Intelligence Chatbot - Type 'quit' to exit, 'reset' to clear state

Capabilities:
  • Search recipes with semantic understanding
  • Compute statistics and rankings
  • Match recipes to your available ingredients
  • Suggest evidence-based ingredient substitutions


🤖 Bot:
Found 5 recipes. Showing top 5:

1. **Baked Pancake Squares**
   Steps: 8
   Ingredients: 3/4 cup milk, 2 tablespoons butter, melted, 1/4 cup egg substitute, 1 tablespoon white sugar, 1 cup all-purpose flour...

2. **Fast and Easy Pancakes**
   Steps: 6
   Ingredients: 2 cups milk, 3/4 cup white sugar, 2 eggs, 1 teaspoon vegetable oil, 1 teaspoon vanilla extract...

3. **Thin and Lacy Pancakes**
   Steps: 9
   Ingredients: 2 cups milk, 2 large eggs, 4 tablespoons unsalted butter, melted and cooled, plus more for brushing, 1/2 teaspoon pure vanilla extract, 1 1/4 cups all-purpose flour...

4. **Buttermilk-Herb Pancakes Topped with Caviar **
   Steps: 20
   Ingredients: 1/2 cup crème fraîche or sour cream, 4 tea

## Evaluation & Test Cases

In [15]:
print("\n" + "="*80)
print("EVALUATION: Test Case 1 - Search with Dietary Restrictions")
print("="*80)

state = ConversationState()
print("\nQuery 1: Find a chocolate recipe")
response = process_query("Find a chocolate recipe", state)
print(f"\nResponse:\n{response[:500]}...")

print("\n\nQuery 2: I cannot have sugar")
response = process_query("I cannot have sugar", state)
print(f"\nResponse:\n{response[:5000]}...")
print(f"\nState after follow-up: {state.to_dict()}")


EVALUATION: Test Case 1 - Search with Dietary Restrictions

Query 1: Find a chocolate recipe

Response:
Found 5 recipes. Showing top 5:

1. **Sorbetto di Cioccolatti**
   Steps: 17
   Ingredients: 7 ounces top quality bittersweet chocolate, finely chopped, 2 cups water, 1 cup sugar...

2. **Chocolate Ganache**
   Steps: 7
   Ingredients: 12 ounces semisweet chocolate (use chips or cut block into small pieces), 8 ounces heavy cream, 2 teaspoons flavoring or liqueur (optional)...

3. **Chocolate Ganache**
   Steps: 5
   Ingredients: 12 ounces heavy cream, 12 ounces semisweet chocolate, chopped...

4...


Query 2: I cannot have sugar

Response:
# Recipe Adaptation: Sorbetto di Cioccolato (Sugar-Free Version)

---

## 1. Substitute Recommendation

### ⚠️ Evidence-Based Assessment: None of the Dataset Suggestions Are Suitable

While the dataset identified **chicken breast**, **margarine**, and **cheddar cheese** as the most statistically common alternatives in recipes without sugar, **none

In [16]:
print("\n" + "="*80)
print("EVALUATION: Test Case 2 - Statistical Query")
print("="*80)

state = ConversationState()
print("\nQuery: How many recipes contain both garlic and onion?")
response = process_query("How many recipes contain both garlic and onion? What percentage is that?", state)
print(f"\nResponse:\n{response}")


EVALUATION: Test Case 2 - Statistical Query

Query: How many recipes contain both garlic and onion?

Response:
Found 12,713 recipes containing all of: garlic, onion.

This represents 10.16% of the 125,164 total recipes in the dataset.


In [18]:
print("\n" + "="*80)
print("EVALUATION: Test Case 3 - Ingredient Matching")
print("="*80)

state = ConversationState()
print("\nQuery: I have onions, tomatoes, and rice. Use all the ingredients to give me the best recipes. No peanuts, at most 6 steps.")
response = process_query("I have onions, tomatoes, and rice. Use all the ingredients to give me the best recipes. No peanuts, at most 6 steps.", state)
print(f"\nResponse:\n{response}")


EVALUATION: Test Case 3 - Ingredient Matching

Query: I have onions, tomatoes, and rice. Use all the ingredients to give me the best recipes. No peanuts, at most 6 steps.

Response:
Top 5 recipes using your ingredients:

1. **Wok Made Rice** (Match Score: 1.0)
   Matched: rice (1 total)
   Missing:  (0 total)
   Steps: 5

2. **QATest2** (Match Score: 0.5)
   Matched: rice (1 total)
   Missing: water (1 total)
   Steps: 1

3. **Sun-dried Tomatoes I** (Match Score: 0.5)
   Matched: rice (1 total)
   Missing: milk (1 total)
   Steps: 6

4. **Homemade Stewed Tomatoes** (Match Score: 0.5)
   Matched: tomatoes (1 total)
   Missing: salt (1 total)
   Steps: 3

5. **QA** (Match Score: 0.5)
   Matched: tomatoes (1 total)
   Missing: salt (1 total)
   Steps: 2



In [19]:
print("\n" + "="*80)
print("EVALUATION: Test Case 4 - Ingredient Substitution")
print("="*80)

state = ConversationState()
print("\nQuery 1: Find banana bread")
response = process_query("Find banana bread", state)
print(f"\nResponse:\n{response[:300]}...")

print("\n\nQuery 2: Replace butter in this recipe")
response = process_query("Replace butter in this recipe", state)
print(f"\nResponse:\n{response[:8000]}...")


EVALUATION: Test Case 4 - Ingredient Substitution

Query 1: Find banana bread

Response:
Found 5 recipes. Showing top 5:

1. **Banana Bread With Chocolate Chips and Chocolate Glaze**
   Steps: 13
   Ingredients: 1 stick unsalted butter, melted and cooled (or 1/2 cup vegetable oil), plus more for the pan, 1 1/4 cups all-purpose flour, 3/4 cup granulated sugar, 1 teaspoon baking powder, 1...


Query 2: Replace butter in this recipe

Response:
# Recipe Adaptation: Banana Bread With Chocolate Chips and Chocolate Glaze
## Replacing: Butter

---

## 1. 🔍 Substitute Recommendation: **Margarine**

Based on the dataset evidence, **margarine** is the most contextually appropriate substitute, appearing in **13 similar recipes with a strong enrichment score of 7.52x**.

> ⚠️ **Important note on the evidence:** While `ground beef` appeared most frequently (21 recipes), it is **categorically unsuitable** for a sweet baking application — its presence likely reflects a data artifact from unrelated re

In [20]:
print("\n" + "="*80)
print("EVALUATION: Test Case 5 - No Match Scenario")
print("="*80)

state = ConversationState()
state.exclude_ingredients = ['egg', 'dairy', 'gluten', 'soy', 'nut']
state.max_steps = 1
print("\nQuery: Find chocolate cake (with extreme constraints)")
print(f"Constraints: {state.to_dict()}")
response = process_query("Find chocolate cake", state)
print(f"\nResponse:\n{response}")


EVALUATION: Test Case 5 - No Match Scenario

Query: Find chocolate cake (with extreme constraints)
Constraints: {'exclude_ingredients': ['egg', 'dairy', 'gluten', 'soy', 'nut'], 'include_ingredients': [], 'max_steps': 1, 'user_ingredients': []}

Response:
Found 5 recipes. Showing top 5:

1. **Pound Cake with Vanilla Ice Cream and Chocolate Sauce**
   Steps: 1
   Ingredients: 4 slices pound cake, 1-inch thick, 1 pint vanilla bean ice cream, Chocolate syrup (recommended: Hershey brand)...

2. **Queen Mother's Cake**
   Steps: 1
   Ingredients: 6 ounces (scant 1 1/2-cups) blanched or unblanched almonds, 6 ounces unsalted butter, plus extra for pan, Fine dry bread crumbs, 6 ounces semisweet chocolate, cut into small pieces, 3/4 cup granulated sugar...

3. **Queen Mother's Cake**
   Steps: 1
   Ingredients: 6 ounces (scant 1 1/2-cups) blanched or unblanched almonds, 6 ounces unsalted butter, plus extra for pan, Fine dry bread crumbs, 6 ounces semisweet chocolate, cut into small pieces, 3/4